**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Performance Engineering & the Roofline Model

The unifying diagram behind every 'why is this slow' conversation in [GPU](../Intro_GPU/README.md), [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb), and [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb): measure your machine's compute roof and memory roof, place your kernels on the chart, and *know* which wall you're hitting before touching a line of code.

## 1. Pre-requisites

[Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) (the CGMA idea), [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) (caches exist).

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def bench(fn, *args, reps=7):
    ts = []
    for _ in range(reps):
        tic = time.perf_counter(); fn(*args); ts.append(time.perf_counter()-tic)
    return min(ts)                                  # min = least OS interference

---
### 🕐 Session 1 of 2 — *Measuring Your Machine's Roofs* (~40 min)
**Goal:** peak FLOP/s from matmul, peak GB/s from streaming — the two ceilings of all performance.
**Feeds into:** Session 2 (placing kernels on the roofline).

---

## 2. Two Ceilings

💡 **Intuition.** Every kernel is limited by one of two machine properties: how fast it can **compute** (FLOP/s, ceiling set by matmul-class code) or how fast it can **move data** (bytes/s, ceiling set by streaming). Which one binds is decided by the kernel's **arithmetic intensity** — FLOPs per byte touched — the same quantity [Intro_GPU](../Intro_GPU/Intro_GPU.ipynb) called CGMA. Below the machine's critical intensity, no cleverness in the arithmetic helps: you are paying for trucks, not workers.

In [ ]:
# roof 1: compute (large matmul → BLAS at near-peak)
# roof 2: memory bandwidth (pure streaming: y = x copy/scale of a cache-busting array)

# YOUR CODE HERE


**What just happened.** Two numbers that between them bound everything this machine can ever do:

| roof | measured | what it limits |
|---|---|---|
| compute | **1415.1 GFLOP/s** | how fast arithmetic happens |
| memory | **41.7 GB/s** | how fast bytes arrive |
| ratio | **34.0 FLOP/byte** | which of the two binds |

**Each benchmark is a probe aimed at one wall, and that is what makes the result a ceiling rather than a data point.** The $2048^3$ matmul has arithmetic intensity $n/6 \approx 341$ FLOP/byte — ten times the critical value — so memory cannot possibly constrain it, and the number measures **compute**. The 256 MB copy has intensity near zero and is far too large for any cache, so it measures **bandwidth**. Neither is a general-purpose benchmark.

**Now read the 34, because the whole workshop turns on it.** For every 4-byte float fetched from main memory, this machine can perform about **136 floating-point operations** before arithmetic becomes the limit. **Touch a number once and do one thing with it, and you are using well under 1% of the machine.** That is not a tuning problem — it is a structural property of the hardware, and faster arithmetic does not touch it.

**The size of that gap is also the defining hardware trend of the last thirty years.** Compute has grown far faster than memory bandwidth, so the critical intensity keeps rising and more code falls on the memory-bound side each generation. **Cache blocking, operator fusion, and mixed precision are all the same move** — they raise intensity rather than speeding up arithmetic, which is the only thing that helps below the critical line.

**Note `bench`'s one real design decision, since it is the methodology rather than a detail.** It returns the **minimum** of seven runs, not the mean. Every source of interference — preemption, cache eviction by another process, thermal throttling — makes a run *slower* and never faster, so **the minimum is the closest estimate of the machine's true capability**. Averaging mixes in noise whose sign you already know.

**Two honest caveats about these specific roofs, both of which matter in Session 2.** First, `peak_flops` was measured **from a 2048-matmul** — so when Session 2 plots a 2048-matmul against it and reports 103%, that is **confirming the definition, not discovering a fact**. Anything reaching ~100% here is circular; an external check against the vendor's published peak would not be.

**Second, 41.7 GB/s is a single-threaded copy.** A multi-threaded streaming benchmark on the same hardware typically reaches two to four times that, so **the true critical intensity is probably lower than 34** and the memory roof drawn here is conservative. **The roofline is a model with measured inputs**, and the inputs carry their own methodology.

**One sanity check worth thirty seconds.** 1415 GFLOP/s in float32 implies AVX-512-class vector FMA across several cores. Divide by your core count and clock speed to get FLOPs per cycle per core; if the answer exceeds what the instruction set can issue, the measurement is wrong somewhere. **A roof you cannot justify from the hardware specification is a roof you should not trust.**

---
### 🕐 Session 2 of 2 — *Kernels on the Roofline* (~40 min)
**Goal:** place real operations on the chart; watch intensity, not effort, decide their fate.
**Builds on:** Session 1.

---

## 3. The Chart That Ends Arguments

In [ ]:
# measure several kernels: achieved FLOP/s vs arithmetic intensity
# saxpy: 2 FLOPs per 12 bytes → intensity 0.167 (hopelessly memory-bound)
# elementwise exp: ~1 'FLOP' per 8 bytes (in truth many flops inside exp — we count 1 op)
# small matmul (fits cache) vs large: same math, different effective intensity

# YOUR CODE HERE


**What just happened.** Four kernels placed on the roofline — and **two of them report more than 100% of their own ceiling**, which is impossible:

| kernel | achieved | % of its ceiling | verdict |
|---|---|---|---|
| saxpy | 1.6 GFLOP/s | 23% | memory-bound, and below its roof |
| exp | 11.9 GFLOP/s | **228%** | **impossible — the model is wrong** |
| matmul 64 (cache-warm) | 117.3 GFLOP/s | 26% | mislabelled (see below) |
| matmul 2048 | 1458.9 GFLOP/s | **103%** | **circular — see below** |

**Nothing can exceed a roof.** A percentage above 100% is not a triumph; it is the model telling you that its **inputs** — your FLOP count or your byte count — are wrong. **Learning to read that as a bug report rather than a result is the most valuable thing in this cell.**

**Diagnose `exp` first, because two errors compound there.** The code charges `np.exp` as **one FLOP per element**, and the comment admits it. A vectorised exponential is really 10–20 arithmetic operations per element, so the achieved GFLOP/s is understated by an order of magnitude *and* the intensity of $1/8$ is understated too. On top of that, `x[:M//4]` is **64 MB** — on a machine with a large L3 it may be partly cache-resident, so fewer bytes come from DRAM than the model assumes. **Both inputs are wrong in the same direction, and 228% is the result.**

**Now `matmul 2048` at 103%, where the cause is different and more subtle.** The compute roof in Session 1 was *measured from a 2048-matmul*. Re-running the same operation and comparing it against that roof is **circular** — 103% is run-to-run variation around a number this very kernel defined. **It confirms the definition rather than discovering a fact**, and reading it as "we beat the machine" would be a mistake.

**And `matmul 64` is mislabelled in a third way.** The printout calls it *memory-bound*, but a $64\times64$ float32 matmul touches only **48 KB** — it fits entirely in L1/L2, so there is essentially **no main-memory traffic at all** and its true intensity is far above the $n/6 = 10.7$ the code assigns. **The model charged it for DRAM traffic that never happened.** Its real limits are call overhead and having too little work to fill the vector units — which is why 26% is a fair description of the *performance* and "memory-bound" is not a fair description of the *cause*.

**With those three caveats stated, the one clean result is the most instructive.** `saxpy` sits at intensity 0.167 with a ceiling of 7 GFLOP/s — versus the machine's 1415 — and reaches 23% of it. **It is memory-bound by construction, and nothing about the arithmetic can help.** Two FLOPs per 12 bytes is the whole story; a perfectly optimised saxpy is still 200× slower than matmul on this hardware.

**The comparison between the two matmuls is the argument the chart exists to make.** Identical mathematics, identical library, sizes 64 and 2048 — and they land in completely different regimes. **Intensity, not algorithm, decided their fate.** That is why "which algorithm is fastest?" is usually the wrong question and "how much reuse does this get per byte?" is the right one.

**So the diagnosis becomes mechanical, and that is the deliverable.** Far below your roof → fix the implementation. Sitting *on* a memory roof → the code is fine, and only **raising intensity** helps: fuse operations, tile for cache ([HW-Accelerated Computing](../Intro_GPU/HW_Accelerated_Computing.ipynb) does this with shared memory), or change algorithm. **Above your roof → recount your FLOPs and bytes**, which is exactly what two rows of this table are telling you to do.

**One correction worth applying if you rerun this.** Count `np.exp` at ~15 FLOPs per element and use an array well beyond L3 for the streaming kernels. **The impossible number should disappear** — and watching it disappear teaches the model's dependence on its inputs better than any explanation.

💡 **Intuition.** The diagnosis is now mechanical: a kernel far *below* its roof has implementation problems (fix the code); a kernel *on* a memory roof can only be helped by **raising its intensity** — fuse operations, tile for cache ([HW-Accelerated Computing's](../Intro_GPU/HW_Accelerated_Computing.ipynb) shared-memory story), or change algorithm. Optimizing a memory-bound kernel's arithmetic is polishing the truck's engine while it waits at the loading dock.

**The habit:** before optimizing anything, compute its intensity on a napkin and place it on this chart. Half of all optimization effort in the wild is spent on the wrong side of the critical intensity.

## 4. Conclusion

Two measured roofs, one intensity axis, every kernel placed — and the fix (better code vs more reuse vs different algorithm) read directly off the chart.

---
## Where next

- [HW-Accelerated Computing](../Intro_GPU/HW_Accelerated_Computing.ipynb) — raising intensity with shared memory.
- [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — rooflines at training scale.